# Ascent AI: Academic Planning Assistant

Ascent AI is a multi-agent system that helps students turn "I don't know what to do next" into a concrete plan: an academic track, a shortlist of target universities, and a prioritized set of next steps.

## Why this exists

Most students don't get enough one-on-one academic guidance. Counselor caseloads are high, sessions are infrequent, and it's easy to end up with a list of interests but no clear path from those interests to an application strategy. Ascent AI is built to close that gap by acting as an always-available advisor that remembers a student's history and revisits their plan as things change.

## How it works

The system is a sequential pipeline of specialized agents, built on Google's Agent Development Kit (ADK) and powered by Gemini:

1. **Profile Agent** - parses a free-form description of the student into a structured profile (interests, grades, constraints, scores) and saves it.
2. **Mapper Agent** - proposes a small set of plausible academic tracks, selects the strongest match, and drafts an action plan.
3. **University Agent** - filters a university database against the student's budget, location preferences, and track to produce a reach/target/safe shortlist, then compiles everything into a single report.
4. **Mentor Agent** - runs independently, in a later session. It retrieves the saved profile and plan, compares them against a new progress update from the student, and returns revised next steps.

State is held in a lightweight in-memory store, which is what lets the Mentor Agent pick up where the initial planning session left off.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("Gemini API key loaded.")
except Exception as e:
    print("Authentication error: add 'GOOGLE_API_KEY' to your Kaggle secrets.")
    raise e


In [ ]:
from typing import Any, Dict, List
import json

from google.genai import types

from google.adk.agents import Agent, LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import AgentTool
from google.adk.tools.tool_context import ToolContext

print("ADK components imported.")

# Retry configuration for transient API errors
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

# Session service (conceptual) - manages session state on behalf of a client
session_service = InMemorySessionService()
print("Session service created.")


def pretty_print_json(data: Any):
    print(json.dumps(data, indent=2, ensure_ascii=False))

print("Helper function ready.")


## Configuring the university database

The `UniversitySearchTool` filters candidates from a hardcoded list, `UNIVERSITY_DB`. This is placeholder data - replace it with the set of universities relevant to your students (for example, a school's historical target list) before using this for real.

Each entry needs to include:

- `name` (string)
- `country` (string)
- `tuition_band` - one of `"low"`, `"medium"`, `"high"`
- boolean flags for supported majors (`has_cs`, `has_ds`, `has_business`, etc.)

The filtering logic in `UniversitySearchTool` reads these fields directly, so a new entry that skips a flag will just be treated as not offering that major.


In [ ]:
# === UniversitySearchTool: custom tool used by the University Agent ===

UNIVERSITY_DB: List[Dict[str, Any]] = [
    {
        "name": "Delft University of Technology (TU Delft)",
        "country": "Netherlands",
        "tuition_band": "high",
        "has_cs": True,
        "has_ds": True,
        "has_business": False,
        "has_psychology": False,
        "notes": "Top-ranked Engineering/Tech university in NL.",
    },
    {
        "name": "University of Amsterdam (UvA)",
        "country": "Netherlands",
        "tuition_band": "medium",
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": True,
        "notes": "Broad programs, strong for Business, CS, and Psychology.",
    },
    {
        "name": "Vrije Universiteit Amsterdam (VU)",
        "country": "Netherlands",
        "tuition_band": "high", # Non-EEA Bachelor's institutional fees often high (e.g., $18k-$22k)
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": True,
        "notes": "Strong in AI, Computer Science, and Medical Sciences.",
    },
    {
        "name": "Eindhoven University of Technology (TU/e)",
        "country": "Netherlands",
        "tuition_band": "high", # Non-EEA Bachelor's institutional fees often high (e.g., ~$18.6k)
        "has_cs": True,
        "has_ds": True,
        "has_business": False,
        "has_psychology": False,
        "notes": "Top-tier technical university, excellent for engineering.",
    },
    {
        "name": "Maastricht University (UM)",
        "country": "Netherlands",
        "tuition_band": "medium", # Non-EEA Bachelor's institutional fees often lower (e.g., ~$10k-14k depending on program)
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": True,
        "notes": "Known for Problem-Based Learning (PBL) and International Business.",
    },
    {
        "name": "University of Groningen (RUG)",
        "country": "Netherlands",
        "tuition_band": "medium", # Non-EEA Bachelor's institutional fees often lower (e.g., ~$13.5k-$19.8k depending on faculty)
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": True,
        "notes": "Comprehensive university with diverse English-taught programs.",
    },
    {
        "name": "Erasmus University Rotterdam",
        "country": "Netherlands",
        "tuition_band": "medium",
        "has_cs": False,
        "has_ds": True,
        "has_business": True,
        "has_psychology": False,
        "notes": "Best known for Finance and Management (Rotterdam School of Management).",
    },
    {
        "name": "Carnegie Mellon University",
        "country": "USA",
        "tuition_band": "high",
        "has_cs": True,
        "has_ds": True,
        "has_business": False,
        "has_psychology": False,
        "notes": "Strong CS/AI program",
    },
    {
        "name": "University of Toronto",
        "country": "Canada",
        "tuition_band": "high",
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": True,
        "notes": "Top global university",
    },
    {
        "name": "University of Waterloo",
        "country": "Canada",
        "tuition_band": "high",
        "has_cs": True,
        "has_ds": True,
        "has_business": False,
        "has_psychology": False,
        "notes": "Excellent for CS/Engineering",
    },
    {
        "name": "National University of Singapore",
        "country": "Singapore",
        "tuition_band": "high",
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": False,
        "notes": "Top Asian tech school",
    },
    {
        "name": "University of Twente (UT)",
        "country": "Netherlands",
        "tuition_band": "medium",
        "has_cs": True,
        "has_ds": False,
        "has_business": False,
        "has_psychology": False,
        "notes": "Strong focus on Engineering and Applied Sciences.",
    },
    {
        "name": "Arizona State University",
        "country": "USA",
        "tuition_band": "medium",
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": False,
        "notes": "Good CS and Data Science options",
    },
    {
        "name": "BITS Pilani",
        "country": "India",
        "tuition_band": "medium",
        "has_cs": True,
        "has_ds": True,
        "has_business": False,
        "has_psychology": False,
        "notes": "Strong for CS and engineering in India",
    },
    {
        "name": "IIT Bombay",
        "country": "India",
        "tuition_band": "medium",
        "has_cs": True,
        "has_ds": True,
        "has_business": False,
        "has_psychology": False,
        "notes": "Premier Indian tech institute",
    },
    {
        "name": "UT Dallas",
        "country": "USA",
        "tuition_band": "medium",
        "has_cs": True,
        "has_ds": True,
        "has_business": True,
        "has_psychology": False,
        "notes": "Popular for CS/data with moderate tuition",
    },
    {
        "name": "Michigan State University",
        "country": "USA",
        "tuition_band": "low",
        "has_cs": True,
        "has_ds": False,
        "has_business": True,
        "has_psychology": True,
        "notes": "Affordable local option",
    },
    {
        "name": "Azimji Premji University",
        "country": "India",
        "tuition_band": "low",
        "has_cs": False,
        "has_ds": False,
        "has_business": True,
        "has_psychology": True,
        "notes": "Business and psychology focused",
    },
]


def UniversitySearchTool(profile: Dict[str, Any], main_track: str) -> dict:
    """
    Custom Tool: Filters universities based on profile constraints and chosen track.

    This function is used by the University Agent to separate universities into:
      - reach
      - target
      - safe

    Args:
        profile: structured student profile (including constraints)
        main_track: the main recommended track/major string

    Returns:
        dict: {
          "status": "success",
          "main_track": "...",
          "reach": [...],
          "target": [...],
          "safe": [...]
        }
    """

    
    
    constraints = profile.get("constraints", {})
    countries = constraints.get("countries")
    budget = constraints.get("budget_band")
    print(budget)

    main_track = main_track.lower()

    MAJOR_CATEGORIES = [
    ("cs", ["computer", "cs", "ai", "software"]),
    ("ds", ["data", "analytics"]),
    ("business", ["business", "management", "entrepreneur"])]

    main_track_flags = {}
    final_uni_list = []

    []

    for flag, keywords in MAJOR_CATEGORIES:
        if any(keyword in main_track for keyword in keywords):
            main_track_flags[flag] = True
    
    for uni in UNIVERSITY_DB:
        if budget == uni['tuition_band'] and countries in uni['country']:
            for key, value in main_track_flags.items():
                if uni.get(f'has_{key}') and uni[f'has_{key}']:
                    final_uni_list.append(uni)
                    break

    reach = final_uni_list[:2]
    target = final_uni_list[2:5]
    safe = final_uni_list[5:]

    result = {
        "status": "success",
        "main_track": main_track,
        "reach": reach,
        "target": target,
        "safe": safe
    }
print("UniversitySearchTool defined.")

In [ ]:
# === Memory Bank: Long-term student profile + plan storage ===

STUDENT_MEMORY: Dict[str, Dict[str, Any]] = {}
PLAN_MEMORY: Dict[str, Dict[str, Any]] = {}


def save_profile_tool(student_id: str, profile: Dict[str, Any]) -> dict:
    """
    Tool: Save or update a student's profile in the memory bank.
    """
    STUDENT_MEMORY[student_id] = profile
    return {"status": "success", "student_id": student_id}


def get_profile_tool(student_id: str) -> dict:
    """
    Tool: Retrieve a stored student profile from the memory bank.
    """
    profile = STUDENT_MEMORY.get(student_id)
    if profile is None:
        return {"status": "error", "error_message": f"No profile found for {student_id}"}
    return {"status": "success", "profile": profile}


def save_plan_tool(student_id: str, plan: Dict[str, Any]) -> dict:
    """
    Tool: Save or update a student's action plan in the memory bank.
    """
    PLAN_MEMORY[student_id] = plan
    return {"status": "success", "student_id": student_id}


def get_plan_tool(student_id: str) -> dict:
    """
    Tool: Retrieve a stored student action plan from the memory bank.
    """
    plan = PLAN_MEMORY.get(student_id)
    if plan is None:
        return {"status": "error", "error_message": f"No plan found for {student_id}"}
    return {"status": "success", "plan": plan}


print("Memory bank tools defined (profiles and plans).")

## Agent architecture

Ascent AI runs as a sequential multi-agent system - each agent completes its task and hands structured output to the next:

1. **Profile Agent** - extracts a structured profile from the student's free-form description and persists it with `save_profile_tool`.
2. **Mapper Agent** - reviews the profile, selects a `main_track`, and generates and saves the action plan.
3. **University Agent** - calls `UniversitySearchTool` with the profile and track, then writes a single Markdown report combining the profile, chosen track, university shortlist, and action plan.
4. **Mentor Agent** - runs separately, outside the main pipeline. It retrieves the stored profile and plan with `get_profile_tool` and `get_plan_tool`, compares them against a new progress update, and returns updated guidance.


In [ ]:
# === 1) Profile Agent ===

profile_agent = LlmAgent(
    name="ProfileAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction = """
    You are the Profile Agent. Your primary function is data standardization and persistence.
    
    You will receive a message containing the unique 'STUDENT_ID' and a 'STUDENT_DESCRIPTION' (free-form text).
    
    Your execution MUST follow these three mandatory steps precisely:
    
    STEP 1: Data Extraction and Validation
    Analyze the provided `STUDENT_DESCRIPTION`. Extract all information and strictly map it to the following JSON schema.
    
    Data Rules:
    * **student\_id:** Must be extracted directly from the message and included in the profile.
    * **name:** If the name is explicitly mentioned, use it. If not, default to the string "Student".
    * **grade & board:** Extract the student's current grade level (as an **integer**) and the academic board (e.g., "CBSE", "ICSE", "IB", "State").
    * **interests & strengths:** Summarize these into concise lists of short strings (e.g., ["coding", "astronomy"]).
    * **constraints:**
        * **countries:** List all countries mentioned as preferences.
        * **budget\_band:** Categorize any budget information into exactly one of the three strings: `"low"`, `"medium"`, or `"high"`.
    * **scores:** Map subjects to numerical scores (e.g., {"Math": 90}). If no scores are mentioned, use an empty map: `{}`.
    
    STEP 2: Persistence (Tool Call)
    Call the 'save_profile_tool' once using a tool function call.
    
    **Arguments:** Pass the extracted `student_id` and the complete profile JSON object as the `profile` argument.
    The tool call is mandatory, regardless of data completeness.
    
    STEP 3: Final Output Generation
    Ignore the response from the tool call. Your final output MUST be ONLY the structured JSON profile.
    
    * Do NOT include any Markdown formatting (like ```json), headings, or extra conversational text.
    * The output structure MUST be wrapped under the key "profile".
    
    Final Output Structure to be returned:
    {
      "profile": {
        "student_id": "...",
        "name": "...",
        "grade": ...,
        "board": "...",
        "interests": [...],
        "strengths": [...],
        "constraints": {
          "countries": [...],
          "budget_band": "low|medium|high"
        },
        "scores": {
          "...": ...
        }
      }
    }
  
    """,
    tools=[save_profile_tool],
)

print("ProfileAgent created.")

In [ ]:
# === 2) Mapper Agent ===

mapper_agent = LlmAgent(
    name="MapperAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
You are the Mapper Agent for Ascent AI.

You must perform two phases strictly:

## PHASE 1: Track Mapping
1. Propose 2–3 academic/career tracks or majors based on the profile.
2. Choose ONE main_track that is most aligned.

## PHASE 2: Action Plan Synthesis & Saving (CRITICAL)
3. Using the chosen main_track and the student's grade/constraints, generate the Plan JSON (summary, short_term_tasks, medium_term_tasks, long_term_tasks).
4. CRITICAL: You MUST call save_plan_tool(student_id, plan_json) with the structured plan JSON you generated.

Return ONLY this JSON: { "tracks": [ ... ], "main_track": "string" }
""",
    # Mapper Agent now needs the Plan saving tool
    tools=[save_plan_tool], 
)
print("MapperAgent created.")

In [ ]:
# === 3) University Agent ===
university_agent = LlmAgent(
    name="UniversityAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
You are the University Agent and **FINAL REPORTER** for Ascent AI.

You must follow these steps:

1. Retrieve the saved plan data using get_plan_tool(student_id).
2. Call UniversitySearchTool(profile, main_track) as a tool to get the shortlist.
3. Generate a single, comprehensive, human-readable **Markdown report** that includes:
   - All data captured from the Profile Agent.
   - The Main Track chosen by the Mapper Agent.
   - The full Reach/Target/Safe University Shortlist returned by the tool.
   - The Action Plan Timeline retrieved in Step 1.
   
**IMPORTANT OUTPUT RULE:**
- **DO NOT** return any JSON or code fences.
- The final response MUST be a single, **human-readable Markdown report** following the structure you defined.
""",
    # Tools: UniversitySearchTool and the memory retrieval tool
    tools=[UniversitySearchTool, get_plan_tool], 
)

print("UniversityAgent created.")

In [ ]:
# === 5) Mentor Agent (Loop Agent) ===

mentor_agent = LlmAgent(
    name="MentorAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
You are the Mentor Agent for Ascent AI. You support long-running, continuous guidance for the student.

You will receive:
- A student_id string
- A progress_update string (student describing what they have done since the last plan)

Your steps:
1. Use get_profile_tool(student_id) and get_plan_tool(student_id) to retrieve the original profile and plan.
2. Compare the student's progress_update with the existing timeline.
3. Generate a single, encouraging, and actionable **Markdown message** for the student.

**IMPORTANT OUTPUT RULE:**
- **DO NOT** return any JSON or code fences.
- The final response MUST be a single, **human-readable Markdown message**.

**Message Structure:**
# Progress Check-in for [Student Name]
---
## Status & Feedback
* **Great Job:** Summarize the student's accomplishments (e.g., Python course, coding club) and relate them to their original track.
* **Next Focus:** Gently remind them of an overdue or crucial next task (e.g., SAT Prep) from the Medium-Term tasks.

## Updated Suggestions (Next 30-90 Days)
Based on your progress, focus on:
* [Refined Step 1 for the next 30 days]
* [Refined Step 2 for the next 30 days]
* [Refined Step 3 for the next 30 days]
""",
    tools=[get_profile_tool, get_plan_tool],
)

print("MentorAgent created.")

In [ ]:
# === Root Sequential Multi-Agent System ===

root_agent = SequentialAgent(
    name="AscentPipeline",
    sub_agents=[
        profile_agent,
        mapper_agent,
        university_agent,
        #action_plan_agent,
    ],
)

print("Root SequentialAgent (AscentPipeline) created.")

# In this ADK version, InMemoryRunner only takes the agent
runner = InMemoryRunner(root_agent)
print("InMemoryRunner created for AscentPipeline.")

# Separate runner for the MentorAgent loop
mentor_runner = InMemoryRunner(mentor_agent)
print("InMemoryRunner created for MentorAgent.")


# === Debug printer: prints each agent turn in the pipeline ===

def debug_print_events(events):
    print("\n==================== AGENT TURNS ====================\n")

    for turn in events:
        who = getattr(turn, "source", "Unknown")
        print(f"{who} >\n")

        text_found = False

        # 1) content.text
        if (
            hasattr(turn, "content")
            and turn.content is not None
            and hasattr(turn.content, "text")
            and turn.content.text
        ):
            print(turn.content.text)
            text_found = True

        # 2) content.parts[].text
        if (
            hasattr(turn, "content")
            and turn.content is not None
            and hasattr(turn.content, "parts")
            and turn.content.parts is not None
        ):
            for p in turn.content.parts:
                if hasattr(p, "text") and p.text:
                    print(p.text)
                    text_found = True

        # 3) delta.text
        if hasattr(turn, "delta") and turn.delta is not None:
            if hasattr(turn.delta, "text") and turn.delta.text:
                print(turn.delta.text)
                text_found = True

        # 4) message str
        if hasattr(turn, "message") and isinstance(turn.message, str):
            print(turn.message)
            text_found = True

        if not text_found:
            print("[No text output]")

        print("\n------------------------------\n")

## Running the demo

The next cell runs the full pipeline end to end: Profile Agent, then Mapper Agent, then University Agent.

Before running it, replace `demo_description` with an actual student description. For good results, include:

- Basic info: name, grade, and academic board
- Interests and strengths
- Constraints: budget and countries of interest
- Any relevant scores or grades


In [ ]:
# ============================================================
# DEMO: FULL MULTI-AGENT PIPELINE RUN (WITH PARSED FINAL JSON)
# ============================================================

import asyncio # Ensure asyncio is imported here

demo_description = """
My name is Dev. I recently completed a **Bachelor of Technology (B.Tech) in Computer Science** from an institute in India.
I love Artificial Intelligence (AI), deep learning, data science, and I'm looking for a specialization in Machine Learning. 
I enjoy reading research papers and competing in coding challenges.
My family can afford medium to high tuition, and I'm focused on pursuing a **Master of Science (M.Sc.) in AI in the Netherland* or Germany.
My scores: Overall GPA equivalent to 8.5/10.0 or 3.6/4.0, with top scores in Algorithms and Linear Algebra. I also have one year of experience as a Data Analyst.
"""

prompt = f"""
STUDENT_ID: student_001

STUDENT_DESCRIPTION:
{demo_description}
"""

# Reset memory for a clean run
if "student_001" in STUDENT_MEMORY:
    del STUDENT_MEMORY["student_001"]
if "student_001" in PLAN_MEMORY:
    del PLAN_MEMORY["student_001"]
print("Memory reset for student_001.")


# Run through full multi-agent pipeline with debug info
# The runner handles all agent calls, tool calls, and sequential execution.
response = await runner.run_debug(prompt)


# ---- 1. FILTERED AGENT TURNS (Show the final clear Markdown output) ----
print("\n==================== AGENT TURNS (FILTERED OUTPUT) ====================\n")
for i, turn in enumerate(response):
    who = getattr(turn, "source", f"Turn {i}")
    
    text = None
    if hasattr(turn, "content") and turn.content is not None:
        if hasattr(turn.content, "text") and turn.content.text:
            text = turn.content.text
        elif hasattr(turn.content, "parts") and turn.content.parts is not None:
            # Join all text parts from the turn
            texts = [p.text for p in turn.content.parts if hasattr(p, "text") and p.text]
            if texts:
                text = ("\n".join(texts))
    
    # CRITICAL FILTER: Only print the final agent's Markdown output
    if text and who == "ActionPlanAgent":
        print(f"--- OUTPUT FROM: {who} ---")
        print(text.strip())
        print("\n------------------------------\n")


# ---- 2. CONFIRMATION OF INTERNAL JSON SAVE (Addressing the error) ----
print("\n===== CONFIRMATION OF INTERNAL JSON SAVE (Plan Memory Check) =====\n")

if PLAN_MEMORY.get("student_001"):
    print("Plan found in PLAN_MEMORY.")
    print("This confirms the ActionPlanAgent successfully called the save_plan_tool.")
    pretty_print_json(PLAN_MEMORY["student_001"])
else:
    print("Plan not found in PLAN_MEMORY.")
    print("This indicates the ActionPlanAgent either did not call save_plan_tool or the tool call failed.")
    # Show the last response content to help debug the failure reason
    last_turn = response[-1]
    print("\n--- Last Agent Response Content (Check for missing tool call) ---")
    print(last_turn)

## Progress updates and the Mentor Agent

This section shows how the Mentor Agent handles a follow-up session. Given a `progress_update` describing what the student has done since their last plan, it retrieves their stored profile and plan and returns revised guidance.

Update `progress_update` below with a new student status before running.


In [ ]:
# ============================================================
# PROGRESS UPDATE: MENTOR AGENT LOOP RUN (WITH PARSED OUTPUT)
# ============================================================

import asyncio

progress_update = """
Hi, this is Dev again. Since the last plan:
- I built a small ML project and uploaded it on GitHub. 
- I am also doing an internship at an AI startup
What should I focus on next?
"""

mentor_prompt = json.dumps(
    {
        "student_id": "student_001",
        "progress_update": progress_update,
    },
    indent=2,
)

# Use the separate runner for the Mentor Agent
mentor_runner = InMemoryRunner(mentor_agent)

# Run the Mentor Agent
mentor_response = await mentor_runner.run_debug(mentor_prompt)

# Poll for the final response, since the runner can return a partial
# result before the MentorAgent has finished producing text.

MAX_WAIT_TIME = 10 # Total time to wait for output (in seconds)
sleep_interval = 1 
elapsed_time = 0

while elapsed_time < MAX_WAIT_TIME:
    # Check if the last response part contains content (text or a function response)
    if hasattr(mentor_response[-1], 'content') and mentor_response[-1].content:
        # Check if any text was successfully streamed. If so, break the loop.
        if hasattr(mentor_response[-1].content, 'text') and mentor_response[-1].content.text:
            break
        elif hasattr(mentor_response[-1].content, 'parts') and mentor_response[-1].content.parts:
             if any(p.text for p in mentor_response[-1].content.parts if hasattr(p, 'text')):
                break
    
    # If no final output yet, pause and wait
    # We re-run the runner here to ensure we get a fresh object if the first call was partial/interrupted
    mentor_response = await mentor_runner.run_debug(mentor_prompt) 
    
    await asyncio.sleep(sleep_interval)
    elapsed_time += sleep_interval
    print(f"[Wait: {elapsed_time}s] Waiting for MentorAgent output...")

# Ensure mentor_response is updated after the loop finishes waiting
mentor_response = await mentor_runner.run_debug(mentor_prompt) 


# ---- Extract the FINAL MentorAgent Markdown Output ----
mentor_last = mentor_response[-1]
mentor_text = None

if hasattr(mentor_last, "content") and mentor_last.content is not None:
    if hasattr(mentor_last.content, "text") and mentor_last.content.text:
        mentor_text = mentor_last.content.text
    elif hasattr(mentor_last.content, "parts") and mentor_last.content.parts is not None:
        texts = [p.text for p in mentor_last.content.parts if hasattr(p, "text") and p.text]
        if texts:
            mentor_text = "\n".join(texts)

print("\n===== FINAL MENTOR AGENT OUTPUT =====\n")
print(mentor_text or "[No final text]")

print("\nMentorAgent successfully demonstrated retrieval and updated suggestions.")

## Design summary

A recap of how the pieces fit together.

### Agents

| Agent | Role | Output |
| --- | --- | --- |
| Profile Agent | Parses raw input into a structured student profile | Saves profile via `save_profile_tool` |
| Mapper Agent | Selects the primary academic track and drafts the plan | Saves plan via `save_plan_tool` |
| University Agent | Filters universities against student constraints and compiles the final report | Markdown report combining profile, track, shortlist, and plan |
| Mentor Agent | Retrieves prior context and updates guidance based on new progress | Markdown check-in message |

### Tools and memory

- `UniversitySearchTool` - deterministic filtering of the university database against student constraints (budget, country, track).
- `save_profile_tool` / `get_profile_tool` / `save_plan_tool` / `get_plan_tool` - a small persistence layer backed by two in-memory dictionaries, `STUDENT_MEMORY` and `PLAN_MEMORY`.

### Continuity across sessions

The Mentor Agent is what makes this more than a one-off exchange: it reads the same memory store the initial pipeline wrote to, so a student's next conversation builds on their history instead of starting from scratch.

### Use of Gemini

Gemini is used differently at each stage of the pipeline: structured extraction for the Profile Agent, reasoning and synthesis for the Mapper and University agents, and natural-language coaching for the Mentor Agent.


In [ ]:
# ============================================================
# Regenerate the final output and write the submission file
# ============================================================

import json

print("Running Ascent AI pipeline to regenerate final output...")

demo_description = """
My name is Dev. I recently completed a **Bachelor of Technology (B.Tech) in Computer Science** from an institute in India.
I love Artificial Intelligence (AI), deep learning, data science, and I'm looking for a specialization in Machine Learning. 
I enjoy reading research papers and competing in coding challenges.
My family can afford medium to high tuition, and I'm focused on pursuing a **Master of Science (M.Sc.) in AI in the Netherland* or Germany.
My scores: Overall GPA equivalent to 8.5/10.0 or 3.6/4.0, with top scores in Algorithms and Linear Algebra. I also have one year of experience as a Data Analyst.
"""

prompt = f"""
STUDENT_ID: student_001
STUDENT_DESCRIPTION:
{demo_description}
"""

# Run the pipeline fully again (Profile -> Mapper -> University -> Plan)
demo_response = await runner.run_debug(prompt)

# Extract final turn (ActionPlanAgent)
last_turn = demo_response[-1]
final_text = None

# Safe text extraction (covers text, parts, delta)
if hasattr(last_turn, "content") and last_turn.content:
    if hasattr(last_turn.content, "text") and last_turn.content.text:
        final_text = last_turn.content.text
    elif hasattr(last_turn.content, "parts") and last_turn.content.parts:
        pieces = [p.text for p in last_turn.content.parts if hasattr(p, "text") and p.text]
        if pieces:
            final_text = "\n".join(pieces)

# Fallback
if final_text is None:
    final_text = "[No final text produced by ActionPlanAgent]"

# Try parsing JSON
json_payload = {}
try:
    json_candidate = final_text[final_text.find("{"):]
    json_payload = json.loads(json_candidate)
    print("Parsed final output JSON.")
except Exception as e:
    print("JSON parsing failed:", e)
    json_payload = {"raw_output": final_text}

# Save output file
output_filename = "Ascent_output.json"
with open(output_filename, "w") as f:
    json.dump(json_payload, f, indent=2, ensure_ascii=False)

print(f"Saved submission file: {output_filename}")
print("Output file is ready to submit.")